In [41]:
import pandas as pd
import numpy as np
import json
import pickle

df = pd.read_csv('faceit_clean-10.csv')

separating features  
separating output for classification model and regression model

In [42]:
label_cols = ['team1_win', 'score_diff']
cat_cols = ['map']
num_cols = [c for c in df.columns if c not in label_cols + cat_cols]

X = df[num_cols + cat_cols]
y = df['team1_win']

# Train / Test split

In [43]:
from sklearn.model_selection import train_test_split, cross_val_score

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing Pipeline

In [45]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ]
)

LogisticRegression model

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

log_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipe.fit(X_train, y_train)

y_pred_lr = log_reg_pipe.predict(X_test)
y_proba_lr = log_reg_pipe.predict_proba(X_test)[:, 1]

cv_lr = cross_val_score(log_reg_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(accuracy_score(y_test, y_pred_lr))

0.7815


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


RandomForest

In [47]:
from sklearn.ensemble import RandomForestClassifier


In [48]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])

rf_pipe.fit(X_train, y_train)

y_pred_rf = rf_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]

cv_rf = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Random Forest:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_rf):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_rf):.3f}')
print(f'CV:        {cv_rf.mean():.3f}')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Random Forest:
Accuracy:  0.778
F1 Score:  0.785
AUC:       0.842
CV:        0.773


GradientBoostingClassifier

finded that these feature break the predictions capping them to not allowing them to go to extreme values

In [49]:
from sklearn.ensemble import GradientBoostingClassifier

In [50]:
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
])

gb_pipe.fit(X_train, y_train)

y_pred_gb = gb_pipe.predict(X_test)
y_proba_gb = gb_pipe.predict_proba(X_test)[:, 1]
cv_gb = cross_val_score(gb_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Gradient Boosting:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_gb):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_gb):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_gb):.3f}')
print(f'CV:        {cv_gb.mean():.3f}')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Gradient Boosting:
Accuracy:  0.768
F1 Score:  0.775
AUC:       0.832
CV:        0.760


# New CallibratedClassifier model

In [51]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

In [52]:
calibrated_gb = CalibratedClassifierCV(gb_pipe, cv=5, method='isotonic')
calibrated_gb.fit(X_train, y_train)
pred = calibrated_gb.predict(X_test)

print(classification_report(y_test, pred))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


              precision    recall  f1-score   support

           0       0.79      0.74      0.77       989
           1       0.76      0.81      0.78      1011

    accuracy                           0.78      2000
   macro avg       0.78      0.77      0.77      2000
weighted avg       0.78      0.78      0.77      2000



# Neural Network

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.callbacks import EarlyStopping

preparing data

In [55]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [56]:
input_dim = X_train_processed.shape[1]

model_clf = Sequential()
model_clf.add(Dense(256, input_shape=(input_dim,)))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.3))
model_clf.add(Dense(128))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.3))
model_clf.add(Dense(64))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.3))
model_clf.add(Dense(32))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.2))
model_clf.add(Dense(1))
model_clf.add(Activation('sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [57]:
model_clf.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [58]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True
)

In [59]:
model_clf.fit(
    X_train_processed, y_train,
    epochs=500,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7159 - loss: 0.5796 - val_accuracy: 0.7744 - val_loss: 0.5150
Epoch 2/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7666 - loss: 0.5376 - val_accuracy: 0.7731 - val_loss: 0.5147
Epoch 3/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7656 - loss: 0.5315 - val_accuracy: 0.7763 - val_loss: 0.5098
Epoch 4/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7725 - loss: 0.5277 - val_accuracy: 0.7725 - val_loss: 0.5105
Epoch 5/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7717 - loss: 0.5257 - val_accuracy: 0.7725 - val_loss: 0.5087
Epoch 6/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7725 - loss: 0.5167 - val_accuracy: 0.7713 - val_loss: 0.5102
Epoch 7/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7747 - loss: 0.5112 - val_accuracy: 0.7731 - val_loss: 0.5097
Epoch 8/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7739 - loss: 0.5168 - val_accu

In [60]:
y_pred_nn_prob = model_clf.predict(X_test_processed).flatten()
y_pred_nn_clf = (y_pred_nn_prob >= 0.5).astype(int)

print('Neural Network Classification:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_nn_clf):.3f}')
print(f'F1 Score: {f1_score(y_test, y_pred_nn_clf):.3f}')
print(f'AUC:      {roc_auc_score(y_test, y_pred_nn_prob):.3f}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Neural Network Classification:
Accuracy: 0.777
F1 Score: 0.787
AUC:      0.840


Classification model results and comparison

In [61]:
results = {
    'Logistic Regression': {
        'acc': accuracy_score(y_test, y_pred_lr),
    },
    'Random Forest': {
        'acc': accuracy_score(y_test, y_pred_rf),
    },
    'Gradient Boosting': {
        'acc': accuracy_score(y_test, y_pred_gb),
    },
    'Neural Network': {
        'acc': accuracy_score(y_test, y_pred_nn_clf)
    }
}

for name, r in results.items():
    print(f'{name:<25} {r["acc"]:>10.3f}')

Logistic Regression            0.781
Random Forest                  0.778
Gradient Boosting              0.768
Neural Network                 0.777


# Saving model

In [62]:
import pickle

CalibratedClassifier

In [63]:
with open('calibrated_gb.pkl', 'wb') as f:
    pickle.dump(calibrated_gb, f)

saving the gradient boosting classifier

In [ ]:
with open('gb_classifier.pkl', 'wb') as f:
    pickle.dump(gb_pipe, f)

saving the neural network

In [ ]:
model_clf.save('nn_classifier.keras')

with open('nn_preprocessor.pkl', 'wb') as f:
  pickle.dump(preprocessor, f)
